# Data Cleaning with Python — Complete Teaching Course
**Level:** Beginner to advanced · **Format:** instructor-led demonstrations, discussion, labs, and capstone.

## Goal
Teach the major dimensions of data cleaning: structure, meaning, types, missingness, validity, consistency, uniqueness, integrity, anomalies, reproducibility, and monitoring. No fixed checklist can cover every domain: medical, financial, geographic, and other specialist data require additional rules from their owners.

**Prerequisites:** basic Python, functions, lists, dictionaries; introductory pandas helpful.
**Suggested delivery:** 8 sessions of 2–3 hours, including lab time. Adjust to learners' experience.

| Session | Topics | Learner outcome |
|---|---|---|
| 1 | Data contracts, profiling, raw preservation; steps 1–4 | Diagnose issues before editing |
| 2 | Types, missing values, validation; steps 5–10 | Explain each correction and review item |
| 3 | Files, encodings, text, locales, units | Parse data without losing meaning |
| 4 | Duplicates, keys, joins, reshaping | Protect row grain and relational integrity |
| 5 | Missingness, outliers, time series | Make defensible decisions about unknown and unusual values |
| 6 | Business rules, leakage, specialist data | Separate cleaning from modeling and domain judgment |
| 7 | Pipelines, tests, audit trails, scale | Build repeatable, monitored transformations |
| 8 | Capstone and assessment | Defend and validate an end-to-end cleaning plan |

All records are synthetic. Run code cells in order; all examples need only pandas plus the notebook environment. The first part builds fundamentals; the second part extends them into an advanced course. Exercises are intentionally separate from worked solutions.

## Setup
Use JupyterLab, VS Code, or Google Colab with Python 3.10+ and pandas. If pandas is missing, run `%pip install pandas` in a separate cell, then restart the kernel. No other third-party package is required.

The examples use explicit date formats and nullable data types. Outputs may look slightly different across pandas versions.

In [1]:
from io import StringIO
from pathlib import Path
import pandas as pd
from IPython.display import display
print("pandas version:", pd.__version__)

pandas version: 2.2.3


## Steps
### 1. Load a deliberately messy dataset
Each row should represent one customer. `customer_id` is the intended unique key. `monthly_spend` is a nonnegative amount in USD; dates use year-month-day. Watch for blanks, placeholder strings, inconsistent categories, duplicate rows, invalid ages, malformed dates, and unusual spending.

In [2]:
csv_text = """Customer ID, Name ,Email,Age,City,Join Date,Monthly Spend
C001, Aisha Khan ,AISHA@example.com,29,lahore,2025-01-12,"$1,200.00"
C002,Bilal Ali,bilal@example.com,34, Karachi ,2025-02-03,$850
C003, Sara Ahmed ,sara@example.com,,ISLAMABAD,2025-02-30,$600
C004,Omar Shah,omar.example.com,200,Lahore,2025-03-01,$950
C005,Hina Noor,hina@example.com,27,N/A,2025-03-10,unknown
C006,Usman Raza,usman@example.com,-5,karachi,not recorded,$400
C007,Maryam Ali,maryam@example.com,41,Islamabad,2025-04-05,$50000
C008,Ali Hassan,ali@example.com,unknown,lahore,2025-04-12,-$50
C009,Noor Fatima, ,31,KHI,2025-05-01,$725
C010,Ahmed Khan,ahmed@example.com,38,LHR,2025-05-17,$900
C002,Bilal Ali,bilal@example.com,34, Karachi ,2025-02-03,$850
"""
# Read all fields as strings so we can inspect the original tokens.
raw = pd.read_csv(StringIO(csv_text), dtype="string", keep_default_na=False)
display(raw)


   Customer ID         Name   ...     Join Date Monthly Spend
0         C001   Aisha Khan   ...    2025-01-12     $1,200.00
1         C002     Bilal Ali  ...    2025-02-03          $850
2         C003   Sara Ahmed   ...    2025-02-30          $600
3         C004     Omar Shah  ...    2025-03-01          $950
4         C005     Hina Noor  ...    2025-03-10       unknown
5         C006    Usman Raza  ...  not recorded          $400
6         C007    Maryam Ali  ...    2025-04-05        $50000
7         C008    Ali Hassan  ...    2025-04-12          -$50
8         C009   Noor Fatima  ...    2025-05-01          $725
9         C010    Ahmed Khan  ...    2025-05-17          $900
10        C002     Bilal Ali  ...    2025-02-03          $850

[11 rows x 7 columns]


### 2. Inspect before changing anything
Ask: What is the row grain? Which columns should be unique? What values are missing or impossible? A blank string is not yet a pandas missing value.

In [3]:
print("Rows and columns:", raw.shape)
raw.info()
print("Exact duplicate rows:", raw.duplicated().sum())
display(raw.describe(include="all").T)


Rows and columns: (11, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Customer ID    11 non-null     string
 1    Name          11 non-null     string
 2   Email          11 non-null     string
 3   Age            11 non-null     string
 4   City           11 non-null     string
 5   Join Date      11 non-null     string
 6   Monthly Spend  11 non-null     string
dtypes: string(7)
memory usage: 748.0 bytes
Exact duplicate rows: 1
              count unique                top freq
Customer ID      11     10               C002    2
 Name            11     10          Bilal Ali    2
Email            11     10  bilal@example.com    2
Age              11     10                 34    2
City             11      9             lahore    2
Join Date        11     10         2025-02-03    2
Monthly Spend    11     10               $850    2


### 3. Normalize headers, whitespace, and missing-value markers
Use a copy, not the raw table. Convert placeholders only when their meaning is known: `N/A` could be a legitimate code in another dataset.

In [4]:
clean = raw.copy(deep=True)
clean.columns = clean.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for col in clean.columns:
    clean[col] = clean[col].str.strip()
clean = clean.replace({"": pd.NA, "N/A": pd.NA, "unknown": pd.NA, "not recorded": pd.NA})
display(clean.isna().sum().rename("missing_cells").to_frame())

               missing_cells
customer_id                0
name                       0
email                      1
age                        2
city                       1
join_date                  1
monthly_spend              1


### 4. Standardize categories and remove exact duplicates
Lowercase emails and resolve known city aliases. Remove exact duplicate records only. Two rows with the same ID but different values would need a conflict-resolution rule; blindly keeping the first could discard a real update.

In [5]:
clean["email"] = clean["email"].str.lower()
clean["city"] = clean["city"].str.lower().replace({"khi": "karachi", "lhr": "lahore"}).str.title()
rows_before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print("Exact duplicates removed:", rows_before - len(clean))
conflicts = clean.loc[clean["customer_id"].duplicated(keep=False)]
assert conflicts.empty, "Resolve conflicting customer IDs before continuing."
display(clean[["customer_id", "city", "email"]])

Exact duplicates removed: 1
  customer_id       city               email
0        C001     Lahore   aisha@example.com
1        C002    Karachi   bilal@example.com
2        C003  Islamabad    sara@example.com
3        C004     Lahore    omar.example.com
4        C005       <NA>    hina@example.com
5        C006    Karachi   usman@example.com
6        C007  Islamabad  maryam@example.com
7        C008     Lahore     ali@example.com
8        C009    Karachi                <NA>
9        C010     Lahore   ahmed@example.com


### 5. Parse numeric values and dates
`errors="coerce"` turns failed conversions into missing values. Count failures; coercion without an audit can hide problems. Removing currency symbols is appropriate here because all amounts are USD with comma thousands separators.

In [6]:
age_text = clean["age"].copy()
date_text = clean["join_date"].copy()
spend_text = clean["monthly_spend"].copy()
clean["age"] = pd.to_numeric(age_text, errors="coerce").astype("Int64")
clean["join_date"] = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
spend_numeric = spend_text.str.replace(r"[$,]", "", regex=True)
clean["monthly_spend"] = pd.to_numeric(spend_numeric, errors="coerce").astype("Float64")
parse_failures = pd.Series({
    "age": (age_text.notna() & clean["age"].isna()).sum(),
    "join_date": (date_text.notna() & clean["join_date"].isna()).sum(),
    "monthly_spend": (spend_text.notna() & clean["monthly_spend"].isna()).sum(),
})
display(parse_failures.rename("parse_failures").to_frame())

               parse_failures
age                         0
join_date                   1
monthly_spend               0


### 6. Apply explicit validity rules
For this exercise, valid ages are 18–100 and spend cannot be negative. These are lesson assumptions, not universal rules. Preserve flags before replacing invalid values. The email pattern is a basic format check, not proof that an address exists.

In [7]:
clean["age_invalid"] = (clean["age"].notna() & ~clean["age"].between(18, 100)).fillna(False)
clean["spend_invalid"] = clean["monthly_spend"].lt(0).fillna(False)
clean["email_needs_review"] = ~clean["email"].str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", na=False)
clean.loc[clean["age_invalid"], "age"] = pd.NA
clean.loc[clean["spend_invalid"], "monthly_spend"] = pd.NA
display(clean.loc[clean["age_invalid"] | clean["spend_invalid"] | clean["email_needs_review"],
                  ["customer_id", "age", "monthly_spend", "age_invalid", "spend_invalid", "email_needs_review"]])

  customer_id   age  ...  spend_invalid  email_needs_review
3        C004  <NA>  ...          False                True
5        C006  <NA>  ...          False               False
7        C008  <NA>  ...           True               False
8        C009    31  ...          False                True

[4 rows x 6 columns]


### 7. Handle missing values deliberately
Keep unknown ages, dates, and spend as missing rather than inventing facts. Use `Unknown` for missing cities so the category can be displayed. Demonstrate median imputation in a **separate column** with an indicator; the original cleaned age stays available.

In machine learning, split train/test data first and learn imputation statistics from the training set only. Whole-dataset imputation can leak information.

In [8]:
clean["city"] = clean["city"].fillna("Unknown")
clean["age_was_missing"] = clean["age"].isna()
age_median = clean["age"].median()
clean["age_imputed_demo"] = clean["age"].astype("Float64").fillna(age_median)
print("Median used for teaching example:", age_median)
display(clean[["customer_id", "age", "age_imputed_demo", "age_was_missing"]])

Median used for teaching example: 32.5
  customer_id   age  age_imputed_demo  age_was_missing
0        C001    29              29.0            False
1        C002    34              34.0            False
2        C003  <NA>              32.5             True
3        C004  <NA>              32.5             True
4        C005    27              27.0            False
5        C006  <NA>              32.5             True
6        C007    41              41.0            False
7        C008  <NA>              32.5             True
8        C009    31              31.0            False
9        C010    38              38.0            False


### 8. Flag outliers without automatically deleting them
Use the interquartile range (IQR) rule: values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR need review. A large purchase may be real. Small samples make these thresholds unstable; use business context before deciding.

In [9]:
q1, q3 = clean["monthly_spend"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
clean["spend_outlier"] = ((clean["monthly_spend"] < lower) | (clean["monthly_spend"] > upper)).fillna(False)
print(f"Review bounds: USD {lower:,.2f} to USD {upper:,.2f}")
display(clean.loc[clean["spend_outlier"], ["customer_id", "monthly_spend"]])

Review bounds: USD 215.62 to USD 1,490.62
  customer_id  monthly_spend
6        C007        50000.0


## Checks
### 9. Validate the result and report remaining issues
A cleaned dataset can still have unknown values and unresolved review items. Assertions test the rules we promised; they do not prove all records are true.

In [10]:
assert len(raw) == 11  # The source remains unchanged.
assert len(clean) == 10
assert clean["customer_id"].notna().all()
assert clean["customer_id"].is_unique
assert clean["age"].dropna().between(18, 100).all()
assert clean["monthly_spend"].dropna().ge(0).all()
assert clean["city"].isin(["Lahore", "Karachi", "Islamabad", "Unknown"]).all()
assert pd.api.types.is_datetime64_any_dtype(clean["join_date"])
print("All validation checks passed.")
display(clean.isna().sum().loc[lambda s: s > 0].rename("remaining_missing").to_frame())
display(clean.head())

All validation checks passed.
               remaining_missing
email                          1
age                            4
join_date                      2
monthly_spend                  2
  customer_id        name  ... age_imputed_demo  spend_outlier
0        C001  Aisha Khan  ...             29.0          False
1        C002   Bilal Ali  ...             34.0          False
2        C003  Sara Ahmed  ...             32.5          False
3        C004   Omar Shah  ...             32.5          False
4        C005   Hina Noor  ...             27.0          False

[5 rows x 13 columns]


### 10. Export the cleaned data and a review queue
Use a separate filename; never overwrite the source. CSV does not preserve pandas dtypes, so downstream readers must parse dates and set types again. A missing required email is kept for review instead of silently removing the customer.

In [11]:
output_dir = Path("data_cleaning_outputs")
output_dir.mkdir(exist_ok=True)
review_mask = (clean["age_invalid"] | clean["spend_invalid"] | clean["email_needs_review"]
               | clean["spend_outlier"] | clean["join_date"].isna())
clean.to_csv(output_dir / "customers_clean.csv", index=False)
clean.loc[review_mask].to_csv(output_dir / "customers_review.csv", index=False)
print("Clean rows:", len(clean), "| Rows needing review:", int(review_mask.sum()))
print("Files saved in:", output_dir.resolve())

Clean rows: 10 | Rows needing review: 6
Files saved in: /workspace/scratch/d2095f4da170/data_cleaning_outputs


## Next Steps
### Practice: clean a new dataset
Work independently before opening the solutions. Keep the source intact.

1. Normalize headers and whitespace, and convert blank values to missing.
2. Remove exact duplicates. Check whether product IDs are unique afterward.
3. Parse USD prices and integer quantities. Flag negative prices or quantities; keep zero quantity.
4. Normalize category labels. Leave missing prices unknown.
5. Calculate inventory value only when both price and quantity are known.
6. Assert unique IDs and nonnegative known values. Explain why missing price must not become zero.

In [12]:
practice_csv = """Product ID, Category ,Price,Quantity
P01, books ,$12.50,3
P02,BOOKS,,2
P03,Toys,$20,-1
P04, toys ,-$5,4
P05,Games,$30,0
P01, books ,$12.50,3
"""
practice_raw = pd.read_csv(StringIO(practice_csv), dtype="string", keep_default_na=False)
display(practice_raw)
# Write your cleaning code here. Use a new variable, practice_clean.


  Product ID  Category    Price Quantity
0        P01     books   $12.50        3
1        P02      BOOKS                2
2        P03       Toys     $20       -1
3        P04      toys      -$5        4
4        P05      Games     $30        0
5        P01     books   $12.50        3


### Instructor solution — reveal after attempting the exercise
The following cell is a worked answer. Stop before running it during independent practice.

In [13]:
practice_clean = practice_raw.copy(deep=True)
practice_clean.columns = practice_clean.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for col in practice_clean.columns:
    practice_clean[col] = practice_clean[col].str.strip()
practice_clean = practice_clean.replace("", pd.NA).drop_duplicates().reset_index(drop=True)
practice_clean["category"] = practice_clean["category"].str.title()
practice_clean["price"] = pd.to_numeric(practice_clean["price"].str.replace("$", "", regex=False), errors="coerce").astype("Float64")
practice_clean["quantity"] = pd.to_numeric(practice_clean["quantity"], errors="coerce").astype("Int64")
for col in ["price", "quantity"]:
    practice_clean[col + "_invalid"] = practice_clean[col].lt(0).fillna(False)
    practice_clean.loc[practice_clean[col + "_invalid"], col] = pd.NA
practice_clean["inventory_value_usd"] = practice_clean["price"] * practice_clean["quantity"]
assert practice_clean["product_id"].is_unique
assert practice_clean["price"].dropna().ge(0).all()
assert practice_clean["quantity"].dropna().ge(0).all()
assert practice_clean.loc[practice_clean["product_id"].eq("P05"), "inventory_value_usd"].iloc[0] == 0
display(practice_clean)

  product_id category  ...  quantity_invalid  inventory_value_usd
0        P01    Books  ...             False                 37.5
1        P02    Books  ...             False                 <NA>
2        P03     Toys  ...              True                 <NA>
3        P04     Toys  ...             False                 <NA>
4        P05    Games  ...             False                  0.0

[5 rows x 7 columns]


### Discussion and recap
- **Why not fill every missing value with zero?** Unknown and zero mean different things.
- **Why preserve the raw table?** To reproduce decisions and recover original values.
- **Why keep a valid outlier?** An unusual value can still be correct.
- **What if duplicate IDs disagree?** Investigate source history and define a rule before selecting a record.
- **What changes for production?** Agree on rules with the data owner, record changes, protect personal data, and monitor quality over time.

**Teaching takeaway:** Inspect → standardize → parse → validate → handle missingness → review anomalies → check → export. Every transformation should have a reason.

# Part II — Comprehensive Cleaning Topics
## A. Define quality before changing data
A clean table is fit for a stated purpose, not merely free of nulls. For every dataset, define the **row grain**, keys, schema, acceptable values, units, date/time conventions, required fields, ownership, and intended use.

| Quality dimension | Question | Example check |
|---|---|---|
| Completeness | Are required facts present? | Order ID cannot be missing |
| Validity | Does the value follow a rule? | Quantity is a whole number ≥ 0 |
| Consistency | Do representations agree? | All prices use one declared currency |
| Uniqueness | Is the intended key unique? | One row per order line |
| Integrity | Do relationships hold? | Every order refers to a known customer |
| Timeliness | Is data recent enough? | Latest expected batch arrived |
| Accuracy | Does it match reality? | Compare with a trusted source |

Validity does not prove accuracy. A plausible age can still be wrong. Start with a written contract, profile the source, propose rules, get domain decisions for ambiguous cases, then transform and validate.

**Decision exercise:** Which fields are required for sending an invoice versus analyzing demand? The same missing address may block one use but not the other.

### A1. Profile more than null counts
Inspect dtypes, sample values, distinct counts, missingness, cardinality, range, frequency, and unexpected combinations. Avoid printing private records in production.

In [14]:
profile = pd.DataFrame({
    "dtype": clean.dtypes.astype(str),
    "missing": clean.isna().sum(),
    "distinct_non_null": clean.nunique(dropna=True),
})
display(profile)
display(clean["city"].value_counts(dropna=False))

                             dtype  missing  distinct_non_null
customer_id                 string        0                 10
name                        string        0                 10
email                       string        1                  9
age                          Int64        4                  6
city                        string        0                  4
join_date           datetime64[ns]        2                  8
monthly_spend              Float64        2                  8
age_invalid                boolean        0                  2
spend_invalid              boolean        0                  2
email_needs_review         boolean        0                  2
age_was_missing               bool        0                  2
age_imputed_demo           Float64        0                  7
spend_outlier              boolean        0                  2
city
Lahore       4
Karachi      3
Islamabad    2
Unknown      1
Name: count, dtype: Int64


## B. Ingestion: files, encodings, and structural errors
Before parsing, check the source filename/version, delimiter, encoding, quoting, header location, decimal separator, row count, and expected columns. Preserve original bytes when provenance matters.

| Source | Common issues | Safe response |
|---|---|---|
| CSV/TSV | Wrong delimiter, quoted line breaks, embedded commas, malformed rows | Configure parser explicitly; reject or quarantine bad records |
| Excel | Multiple sheets, merged headers, formulas, date serials, hidden rows | Identify the intended sheet/range; verify cached formula values and workbook date system |
| JSON/API | Nested arrays, missing fields, pagination, schema drift | Normalize at a declared grain; retain source identifiers |
| SQL | Null semantics, timezone conversion, duplicates introduced by joins | Check source grain and row counts before and after joins |
| Logs | Broken lines, mixed formats, duplicates from retries | Keep source line/event IDs and a reject queue |

Do not silently use `on_bad_lines="skip"`: report rejected counts and preserve the records. Excel examples require a suitable engine, such as openpyxl, if you adapt this course to `.xlsx` files.

### B1. Preserve leading zeros and parse locale numbers
IDs are labels, not quantities. A postal code or account number can lose meaning if parsed as an integer. Read them as strings.

In [15]:
sample_csv = "postal_code;amount_eur\n00123;1.234,50\n00456;80,00\n"
locale_data = pd.read_csv(StringIO(sample_csv), sep=";", dtype={"postal_code": "string"},
                          decimal=",", thousands=".")
assert locale_data.loc[0, "postal_code"] == "00123"
assert locale_data.loc[0, "amount_eur"] == 1234.5
display(locale_data)
# For files: pd.read_csv(path, encoding="utf-8", encoding_errors="strict")

  postal_code  amount_eur
0       00123      1234.5
1       00456        80.0


### B2. Decode bytes deliberately
UTF-8 and legacy encodings are different contracts. Never ignore decoding errors without investigating: dropped bytes can alter names or keys.

In [16]:
legacy_bytes = "Café".encode("cp1252")
try:
    legacy_bytes.decode("utf-8")
except UnicodeDecodeError:
    print("UTF-8 rejected these bytes; use the source-declared encoding.")
print(legacy_bytes.decode("cp1252"))

UTF-8 rejected these bytes; use the source-declared encoding.
Café


## C. Text, categories, and numeric semantics
Text cleaning must preserve meaning. Lowercasing can help matching but damage case-sensitive IDs. Accents, Arabic characters, apostrophes, and punctuation may be essential. Keep the original value and build a separate matching key when needed.

### C1. Unicode, invisible whitespace, and matching keys
NFC normalizes canonically equivalent Unicode. NFKC also changes compatibility characters; use it only when those changes are acceptable. Do not strip accents or Arabic diacritics automatically.

In [17]:
import unicodedata
names = pd.Series(["  Café\u00a0Ali  ", "Cafe\u0301 Ali", "O’Neil"], dtype="string")
name_key = names.map(lambda x: unicodedata.normalize("NFC", x)).str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
display(pd.DataFrame({"original": names, "matching_key": name_key}))
assert name_key.iloc[0] == name_key.iloc[1]

       original matching_key
0    Café Ali       café ali
1     Café Ali     café ali
2        O’Neil       o’neil


### C2. Controlled categories and unknown labels
Map only approved aliases. A mapping that returns missing for an unrecognized value should produce a review flag, not an invented category.

In [18]:
status_raw = pd.Series([" paid ", "PAID", "pending", "pendng", "cancelled"], dtype="string")
status_key = status_raw.str.strip().str.casefold()
status_map = {"paid": "Paid", "pending": "Pending", "cancelled": "Cancelled"}
status_clean = status_key.map(status_map)
display(pd.DataFrame({"raw": status_raw, "clean": status_clean,
                      "needs_review": status_key.notna() & status_clean.isna()}))

         raw      clean  needs_review
0      paid        Paid         False
1       PAID       Paid         False
2    pending    Pending         False
3     pendng        NaN          True
4  cancelled  Cancelled         False


### C3. Boolean parsing and integer validation
`astype(bool)` treats nonempty text such as "False" as true. Map explicit tokens. Validate whole numbers before converting to nullable integers.

In [19]:
boolean_raw = pd.Series(["Yes", "no", "TRUE", "0", "maybe", None], dtype="string")
boolean_clean = boolean_raw.str.strip().str.lower().map({"yes": True, "true": True, "1": True,
                                                        "no": False, "false": False, "0": False}).astype("boolean")
display(pd.DataFrame({"raw": boolean_raw, "parsed": boolean_clean}))
quantities = pd.to_numeric(pd.Series(["3", "2.5", "bad", None]), errors="coerce")
fractional = quantities.notna() & quantities.mod(1).ne(0)
quantity_integer = quantities.mask(fractional).astype("Int64")
display(pd.DataFrame({"integer": quantity_integer, "fractional_rejected": fractional}))

     raw  parsed
0    Yes    True
1     no   False
2   TRUE    True
3      0   False
4  maybe    <NA>
5   <NA>    <NA>
   integer  fractional_rejected
0        3                False
1     <NA>                 True
2     <NA>                False
3     <NA>                False


### C4. Units, currencies, and precision
Normalize units using known metadata. Never guess units from magnitude alone. Currency conversion requires a dated exchange-rate policy; rounding is a separate decision.

In [20]:
weights = pd.DataFrame({"value": [70.0, 154.0, 65.0], "unit": ["kg", "lb", "unknown"]})
weights["kg"] = weights["value"] * weights["unit"].map({"kg": 1.0, "lb": 0.45359237})
display(weights)
from decimal import Decimal, ROUND_HALF_UP
invoice_total = (Decimal("19.99") * Decimal("3")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print("Exact decimal invoice total:", invoice_total)

   value     unit         kg
0   70.0       kg  70.000000
1  154.0       lb  69.853225
2   65.0  unknown        NaN
Exact decimal invoice total: 59.97


## D. Missing data: mechanism, meaning, and treatment
Missing values include absent records, empty fields, placeholders, failed parses, invalid values, and structural not-applicable fields. Keep these causes separate when they affect decisions.

| Mechanism | Meaning | Teaching example |
|---|---|---|
| MCAR | Missingness is unrelated to observed or unobserved values | Random sensor transmission loss under a plausible mechanism |
| MAR | Missingness depends on observed variables | Response differs by recorded age group |
| MNAR | Missingness depends on the missing value itself, even after conditioning | High earners avoid reporting income |

These mechanisms are assumptions; a null-count table cannot establish them. Compare missingness across groups and consider collection processes.

**Options:** retain missing values; recover from an authoritative source; exclude records for a justified analysis; impute a constant, group statistic, or model; report sensitivity to alternatives. Each can change bias and uncertainty. Forward fill belongs to a time-ordered, entity-specific process with limits; it does not belong to arbitrary rows.

### D1. Group-aware imputation with explicit fallback
This is a descriptive exercise. For prediction, fit group medians and fallbacks on training data only. Preserve a flag and original column.

In [21]:
measurements = pd.DataFrame({"group": ["A", "A", "B", "B", "C"],
                             "value": [10.0, None, 30.0, 50.0, None]})
measurements["was_missing"] = measurements["value"].isna()
group_median = measurements.groupby("group")["value"].transform("median")
measurements["filled_demo"] = measurements["value"].fillna(group_median).fillna(measurements["value"].median())
display(measurements)
print("Compare distributions and group impacts before adopting imputation.")

  group  value  was_missing  filled_demo
0     A   10.0        False         10.0
1     A    NaN         True         10.0
2     B   30.0        False         30.0
3     B   50.0        False         50.0
4     C    NaN         True         30.0
Compare distributions and group impacts before adopting imputation.


## E. Duplicate records, keys, and entity resolution
Distinguish exact duplicates, repeated business keys, legitimate repeated events, and different records representing the same entity. A transaction retry may be a duplicate; two purchases by one customer usually are not.

For near duplicates, generate candidate pairs using blocking (for example, country + name initial), compare several fields, choose thresholds using labeled examples, and retain uncertain pairs for review. Fuzzy name similarity alone is insufficient. False merges can be more costly than missed merges.

### E1. Deterministic conflict resolution
Assume the business owner explicitly authorizes selecting the latest complete snapshot by update timestamp. Sort stably, reject timestamp ties for a key, and preserve discarded versions. This rule would be wrong for partial updates.

In [22]:
versions = pd.DataFrame({"customer_id": ["C1", "C1", "C2"], "city": ["Lahore", "Karachi", "Lahore"],
                         "updated_at": ["2025-01-01", "2025-02-01", "2025-01-03"]})
versions["updated_at"] = pd.to_datetime(versions["updated_at"], utc=True)
assert not versions.duplicated(["customer_id", "updated_at"]).any(), "Resolve ties first"
ordered = versions.sort_values(["customer_id", "updated_at"], kind="stable")
latest = ordered.drop_duplicates("customer_id", keep="last")
superseded = ordered.loc[~ordered.index.isin(latest.index)]
display(latest)
display(superseded)

  customer_id     city                updated_at
1          C1  Karachi 2025-02-01 00:00:00+00:00
2          C2   Lahore 2025-01-03 00:00:00+00:00
  customer_id    city                updated_at
0          C1  Lahore 2025-01-01 00:00:00+00:00


### E2. Fuzzy matching is a review aid
Use the standard library to illustrate similarity, not automatic merging. Real entity matching also needs domain-specific evaluation and privacy controls.

In [23]:
from difflib import SequenceMatcher
pairs = [("Aisha Khan", "Ayesha Khan"), ("Ali Khan", "Ali Khan")]
for left, right in pairs:
    score = SequenceMatcher(None, left.casefold(), right.casefold()).ratio()
    print(left, "|", right, "| similarity:", round(score, 3))
print("Even identical names do not prove identical people.")

Aisha Khan | Ayesha Khan | similarity: 0.857
Ali Khan | Ali Khan | similarity: 1.0
Even identical names do not prove identical people.


## F. Joins, relationships, and table shape
Define grain before every merge or reshape. A many-to-many join can multiply rows and inflate totals. Check key nulls, key uniqueness on the appropriate side, unmatched rows, output row counts, and aggregate reconciliation. pandas can match null join keys to one another; reject or isolate them when that is not intended.

### F1. Validate a merge and find orphan keys
Here the customer lookup must be unique, while many orders may belong to one customer.

In [24]:
orders = pd.DataFrame({"order_id": [1, 2, 3], "customer_id": ["C1", "C1", "C9"], "amount": [10, 20, 30]})
customers = pd.DataFrame({"customer_id": ["C1", "C2"], "region": ["North", "South"]})
assert orders["customer_id"].notna().all() and customers["customer_id"].notna().all()
joined = orders.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(orders)
assert joined["amount"].sum() == orders["amount"].sum()
display(joined)
display(joined.loc[joined["_merge"].eq("left_only")])

   order_id customer_id  amount region     _merge
0         1          C1      10  North       both
1         2          C1      20  North       both
2         3          C9      30    NaN  left_only
   order_id customer_id  amount region     _merge
2         3          C9      30    NaN  left_only


### F2. Reshape wide data without hiding duplicates
`pivot` rejects duplicate index/column combinations. `pivot_table` aggregates them; do not use it to conceal a grain error.

In [25]:
wide = pd.DataFrame({"store": ["A", "B"], "jan": [10, 20], "feb": [12, 18]})
long = wide.melt(id_vars="store", var_name="month", value_name="sales")
assert not long.duplicated(["store", "month"]).any()
restored = long.pivot(index="store", columns="month", values="sales")
display(long)
display(restored)

  store month  sales
0     A   jan     10
1     B   jan     20
2     A   feb     12
3     B   feb     18
month  feb  jan
store          
A       12   10
B       18   20


### F3. Flatten nested JSON at the intended grain
One parent can contain many child items. Preserve the parent key so expansion does not lose relationships.

In [26]:
payload = [{"order_id": "O1", "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]},
           {"order_id": "O2", "items": [{"sku": "A", "qty": 3}]}]
items = pd.json_normalize(payload, record_path="items", meta="order_id")
assert not items.duplicated(["order_id", "sku"]).any()
display(items)

  sku  qty order_id
0   A    2       O1
1   B    1       O1
2   A    3       O2


## G. Dates, time zones, and time series
Ambiguous dates such as `03/04/2025` need a source convention; do not guess. Distinguish event time from ingestion time, timestamps from calendar dates, and absent measurements from zeros. Handle daylight-saving gaps and repeated local times explicitly. Daylight-saving examples require a source timezone and often additional business context.

### G1. Convert local times to a common instant
A naive timestamp has no timezone. Localize using the declared source timezone, then convert. Do not label local times UTC using `utc=True` unless the source is actually UTC.

In [27]:
local_time = pd.Series(pd.to_datetime(["2025-01-10 09:00", "2025-01-10 10:00"]))
utc_time = local_time.dt.tz_localize("Asia/Karachi").dt.tz_convert("UTC")
display(pd.DataFrame({"karachi_local": local_time, "utc": utc_time}))
# For DST-observing zones, consider ambiguous="raise", nonexistent="raise".


        karachi_local                       utc
0 2025-01-10 09:00:00 2025-01-10 04:00:00+00:00
1 2025-01-10 10:00:00 2025-01-10 05:00:00+00:00


### G2. Detect gaps and bound forward filling
Reindex within each sensor/entity in a real dataset. This single-sensor example fills at most one missing observation; it does not fill from future values.

In [28]:
sensor = pd.Series([10.0, 12.0, 18.0], index=pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-05"]), name="reading")
calendar = pd.date_range(sensor.index.min(), sensor.index.max(), freq="D")
aligned = sensor.reindex(calendar)
display(pd.DataFrame({"observed": aligned, "limited_forward_fill": aligned.ffill(limit=1), "gap": aligned.isna()}))
# Interpolation assumes a process between measurements; justify it before use.


            observed  limited_forward_fill    gap
2025-01-01      10.0                  10.0  False
2025-01-02      12.0                  12.0  False
2025-01-03       NaN                  12.0   True
2025-01-04       NaN                   NaN   True
2025-01-05      18.0                  18.0  False


## H. Outliers, anomalies, and corrupted observations
Use domain bounds, percentiles, IQR, robust median-based measures, plots, temporal changes, and multivariate checks as complementary tools. Skewed distributions and small groups need care. A value can be individually plausible but impossible in combination.

**Treatment choices:** retain and flag, correct from source, quarantine, transform for an analysis, or cap under an explicit policy. Capping and log transforms change the analysis; they do not recover truth. Never remove a point merely because it changes a result you prefer.

### H1. Robust anomaly flag with a zero-MAD guard
Median absolute deviation (MAD) is less affected by extreme points than the standard deviation. The threshold here is an illustrative heuristic, not a universal detector.

In [29]:
values = pd.Series([10.0, 11, 12, 12, 13, 14, 200])
median = values.median()
mad = (values - median).abs().median()
if mad == 0:
    robust_score = pd.Series(float("nan"), index=values.index)
    print("MAD is zero: use a domain rule or another detector.")
else:
    robust_score = 0.6745 * (values - median) / mad
display(pd.DataFrame({"value": values, "robust_score": robust_score, "review": robust_score.abs().gt(3.5)}))

   value  robust_score  review
0   10.0       -1.3490   False
1   11.0       -0.6745   False
2   12.0        0.0000   False
3   12.0        0.0000   False
4   13.0        0.6745   False
5   14.0        1.3490   False
6  200.0      126.8060    True


## I. Cross-field rules and domain-specific cleaning
Examples include start ≤ end, delivery ≥ order date, subtotal + tax = total within a defined rounding tolerance, and part quantities that reconcile with a whole. Validate combinations, not just individual columns.

| Domain | Additional concerns |
|---|---|
| Finance | Currency, decimal precision, reversals, refunds, reconciliation; negative amounts can be valid |
| Healthcare | Units, reference ranges, patient identity, clinical terminology, privacy; use clinical expertise |
| Geography | Latitude/longitude ranges, swapped coordinates, coordinate reference system, address conventions |
| Survey | Skip logic, not-applicable responses, reverse-coded questions, sampling weights |
| Images/audio/text | Corrupt files, metadata, duplicate content, language, labels, consent, train/test overlap |
| Web/event data | Bots, retries, event ordering, session rules, identity changes, late arrivals |

**Exercise:** Explain why a negative refund is valid but a negative customer age is not. Never reuse a generic nonnegative rule across all numeric columns.

### I1. Cross-field validity and review reasons
Preserve records and describe each failure. In larger systems, store one issue per record/rule in a separate audit table.

In [30]:
events = pd.DataFrame({"id": [1, 2, 3], "start": ["2025-01-02", "2025-02-03", None],
                       "end": ["2025-01-05", "2025-02-01", "2025-03-01"]})
for col in ["start", "end"]:
    events[col] = pd.to_datetime(events[col], errors="coerce")
events["dates_missing"] = events[["start", "end"]].isna().any(axis=1)
events["end_before_start"] = events["end"].lt(events["start"])
display(events)

   id      start        end  dates_missing  end_before_start
0   1 2025-01-02 2025-01-05          False             False
1   2 2025-02-03 2025-02-01          False              True
2   3        NaT 2025-03-01           True             False


## J. Cleaning versus preprocessing and data leakage
Cleaning establishes trustworthy records. Scaling, one-hot encoding, feature selection, and transformations prepare data for a particular model. They may follow cleaning but are not interchangeable.

For predictive work: split data by the correct unit and time boundary; fit imputers, scalers, category vocabularies, outlier thresholds, and learned corrections on training data only; apply those learned parameters to validation/test data unchanged. Group or time splits may be necessary to avoid repeated entities or future data leaking across boundaries.

Do not use future outcomes to resolve current records. Do not discard valid test values merely because they are unusual relative to training. Evaluate quality and model outcomes across relevant groups.

### J1. Fit imputation on training data only
The held-out value 1000 must not change the training median.

In [31]:
train = pd.Series([10.0, 20.0, None])
test = pd.Series([None, 1000.0])
training_median = train.median()
train_filled = train.fillna(training_median)
test_filled = test.fillna(training_median)
assert test_filled.iloc[0] == 15.0
print("Training median:", training_median)
display(pd.DataFrame({"test_original": test, "test_filled": test_filled}))

Training median: 15.0
   test_original  test_filled
0            NaN         15.0
1         1000.0       1000.0


## K. Reusable pipelines, tests, and auditability
Put transformations into functions that return new data plus issue reports. Store rule versions, source identifiers, run timestamps, row counts, rejection reasons, and output locations. Keep raw, cleaned, and review data separate. Define rerun behavior and idempotence: applying a normalization twice should usually equal applying it once.

Tests should cover normal cases, nulls, invalid tokens, empty inputs, duplicate keys, unexpected columns, boundary values, and schema changes. A failing check should block publication or quarantine records under a documented policy.

### K1. A tested, idempotent normalizer with a schema gate
This small pipeline shows input validation, raw preservation, explicit parsing, issue reporting, and idempotence. It intentionally rejects unexpected columns instead of silently dropping them.

In [32]:
def normalize_products(frame):
    expected = {"product_id", "category", "price"}
    if set(frame.columns) != expected:
        raise ValueError(f"Expected exactly {sorted(expected)}")
    result = frame.copy(deep=True)
    for col in ["product_id", "category"]:
        result[col] = result[col].astype("string").str.strip().replace("", pd.NA)
    result["category"] = result["category"].str.casefold()
    price_tokens = result["price"].astype("string").str.strip().replace("", pd.NA)
    result["price"] = pd.to_numeric(price_tokens, errors="coerce").astype("Float64")
    issues = pd.DataFrame({
        "missing_id": result["product_id"].isna(),
        "duplicate_id": result["product_id"].duplicated(keep=False) & result["product_id"].notna(),
        "bad_price_token": price_tokens.notna() & result["price"].isna(),
        "negative_price": result["price"].lt(0).fillna(False),
    }, index=result.index)
    return result, issues

fixture = pd.DataFrame({"product_id": [" A ", "B", "C"], "category": [" BOOKS ", "Toys", None], "price": ["12", "bad", "-1"]})
fixture_copy = fixture.copy(deep=True)
normalized, issues = normalize_products(fixture)
pd.testing.assert_frame_equal(fixture, fixture_copy)
normalized_again, _ = normalize_products(normalized)
pd.testing.assert_frame_equal(normalized, normalized_again)
assert issues["bad_price_token"].sum() == 1
assert issues["negative_price"].sum() == 1
empty, _ = normalize_products(fixture.iloc[:0])
assert empty.empty
try:
    normalize_products(fixture.assign(unexpected=1))
except ValueError:
    print("Schema drift rejected as expected.")
else:
    raise AssertionError("Schema drift must be rejected")
display(normalized)
display(issues)
print("Pipeline checks passed.")

Schema drift rejected as expected.
  product_id category  price
0          A    books   12.0
1          B     toys   <NA>
2          C     <NA>   -1.0
   missing_id  duplicate_id  bad_price_token  negative_price
0       False         False            False           False
1       False         False             True           False
2       False         False            False            True
Pipeline checks passed.


## L. Scaling, privacy, exports, and quality monitoring
- **Scale:** prefer vectorized operations and explicit dtypes; read only necessary columns; use chunked reads when appropriate. Global deduplication, quantiles, and cross-chunk joins need shared state or a database/distributed design. Cleaning each chunk separately does not guarantee globally correct results.
- **Performance:** measure memory and time before optimization. Use SQL pushdown or suitable tools such as DuckDB, Polars, or distributed engines when the workload requires them; preserve the same quality contract.
- **Privacy:** use synthetic teaching data; minimize identifiers; control access, retention, and logging. Hashing identifiers alone does not guarantee anonymity. Keep review queues protected.
- **Exports:** CSV loses types and timezone metadata. Parquet preserves more schema but requires a compatible engine. Document units, null conventions, and schema alongside any file. Spreadsheet consumers can interpret values starting with `=`, `+`, `-`, or `@` as formulas; use a consumer-specific safe export policy without corrupting legitimate source data.
- **Monitoring:** track source freshness, row counts, null rates, invalid rates, duplicate rates, orphan keys, and distributions. Define thresholds and owners. Distinguish a change in real behavior from an ingestion problem; a distribution shift is a signal, not automatic proof of bad data.
- **Operations:** publish atomically, version rules and datasets, support rollback, alert on material failures, and specify handling for late-arriving records and corrected source batches.


### L1. Compute a small quality scorecard
Show denominators with rates. Do not average all quality dimensions into a single score that hides critical failures.

In [33]:
scorecard = pd.DataFrame([
    {"check": "Missing email", "affected_rows": int(clean["email"].isna().sum()), "total_rows": len(clean)},
    {"check": "Email requires review", "affected_rows": int(clean["email_needs_review"].sum()), "total_rows": len(clean)},
    {"check": "Spend outlier", "affected_rows": int(clean["spend_outlier"].sum()), "total_rows": len(clean)},
])
scorecard["rate_pct"] = (100 * scorecard["affected_rows"] / scorecard["total_rows"]).round(1)
display(scorecard)
print("Missing email is a subset of email review; do not add overlapping counts.")

                   check  affected_rows  total_rows  rate_pct
0          Missing email              1          10      10.0
1  Email requires review              2          10      20.0
2          Spend outlier              1          10      10.0
Missing email is a subset of email review; do not add overlapping counts.


# Part III — Labs, Capstone, and Assessment
## Progressive labs
| Lab | Task | Evidence to submit |
|---|---|---|
| 1: Inspection | Profile an unfamiliar file; propose a contract | Grain, keys, dtypes, null and category report |
| 2: Parsing | Handle IDs, locale numbers, dates, Boolean tokens | Original tokens, parsed values, failure counts |
| 3: Missingness | Compare retain/drop/group-median decisions | Assumptions and impact by group |
| 4: Identity | Separate exact duplicates from conflicting keys | Duplicate report and approved resolution rule |
| 5: Relationships | Merge transactions to customers safely | Cardinality checks, orphan report, total reconciliation |
| 6: Time | Normalize timezones and detect entity-specific gaps | Declared timezone, gap report, fill justification |
| 7: Pipeline | Build functions and a review queue | Tests, audit log, repeatable outputs |

## Capstone: messy sales orders
Use the synthetic tables below. Treat amounts as USD. Assume one row per order ID and the customer table is a lookup. Do not invent missing customer records or quantities.

**Tasks:**
1. Write a contract: grain, keys, fields, required values, units, dates, and allowed statuses.
2. Preserve raw data; inspect and quantify problems before editing.
3. Standardize text and parse amounts, quantities, dates, and statuses.
4. Identify exact duplicates and conflicting order IDs. Do not resolve conflicts without stating an assumption.
5. Keep unknown, invalid, and not-applicable values distinguishable; flag review cases.
6. Join to customers using cardinality validation; report unmatched IDs.
7. Validate date order and reconcile known quantities × unit price with reported totals under an explicit tolerance.
8. Keep any unresolved records in a protected review queue rather than silently deleting them.
9. Build a function, tests, a quality report, and separate exports. Explain row-count and amount changes.
10. Propose monitoring thresholds and identify who must decide ambiguous cases.


In [34]:
capstone_orders_raw = pd.DataFrame({
    "order_id": ["O01", " O02 ", "O03", "O03", "O04", "O05", "O06", "O07"],
    "customer_id": ["C1", "C2", "C1", "C1", "C9", "C2", "C3", "C1"],
    "order_date": ["2025-01-01", "2025-01-03", "2025-02-30", "2025-02-30", "2025-01-05", "2025-01-06", "2025-01-07", "2025-01-08"],
    "ship_date": ["2025-01-02", "2025-01-02", "", "", "2025-01-07", "2025-01-08", "", "2025-01-09"],
    "quantity": ["2", "3", "1", "1", "-1", "unknown", "2.5", "1"],
    "unit_price": ["$10", "$20", "$30", "$30", "$40", "$10", "$15", "$10000"],
    "reported_total": ["20", "60", "30", "30", "40", "0", "37.50", "10000"],
    "status": [" paid ", "PAID", "pending", "pending", "paid", "cancelled", "pendng", "paid"],
})
capstone_customers_raw = pd.DataFrame({"customer_id": ["C1", "C2", "C3"], "city": ["LHR", "Karachi", " Islamabad "]})
display(capstone_orders_raw)
display(capstone_customers_raw)
# Your work: define a contract, transformation functions, review rules, and checks.


  order_id customer_id  order_date  ... unit_price reported_total     status
0      O01          C1  2025-01-01  ...        $10             20      paid 
1     O02           C2  2025-01-03  ...        $20             60       PAID
2      O03          C1  2025-02-30  ...        $30             30    pending
3      O03          C1  2025-02-30  ...        $30             30    pending
4      O04          C9  2025-01-05  ...        $40             40       paid
5      O05          C2  2025-01-06  ...        $10              0  cancelled
6      O06          C3  2025-01-07  ...        $15          37.50     pendng
7      O07          C1  2025-01-08  ...     $10000          10000       paid

[8 rows x 8 columns]
  customer_id         city
0          C1          LHR
1          C2      Karachi
2          C3   Islamabad 


## Instructor checkpoints for the capstone
Use after learners attempt the task. These are diagnostic expectations, not a universal cleaning policy.

- There are eight raw order rows and seven after removing the one exact duplicate O03.
- O02 ships before it was ordered; O03 has an impossible order date and missing shipping date.
- C9 has no lookup customer. O04 has negative quantity; O05 quantity is unknown; O06 quantity is fractional under a whole-unit contract.
- `pendng` is an unapproved status token unless the owner authorizes a correction.
- O07 has a large amount; it deserves review, not automatic deletion.
- Cancellation may explain a missing shipment or zero reported total. Decide rules using status and the business definition.
- Do not compute a zero total from unknown quantity. Do not compare amount totals across different retained populations without explanation.

## Assessment rubric
| Criterion | Points |
|---|---:|
| Clear contract and assumptions | 15 |
| Profiling and preservation of raw data | 15 |
| Correct, explainable transformations | 20 |
| Duplicates, relationships, and reconciliation | 15 |
| Review queue, audit trail, and privacy | 10 |
| Meaningful tests and repeatability | 15 |
| Communication of unresolved issues and monitoring | 10 |
| **Total** | **100** |

## Knowledge check — discussion questions
1. Can a dataset have no nulls and still be inaccurate?
2. How do an unknown price and a zero price differ?
3. When does removing duplicates destroy valid events?
4. Why can a join increase sales totals?
5. Why must a valid outlier sometimes remain?
6. Which transformations must be learned on training data only?
7. What evidence is needed before inferring a missingness mechanism?
8. What makes a pipeline idempotent, and what does that not guarantee?

**Instructor answer cues:** Yes—plausible false values; unknown is absent knowledge; repeated events can be legitimate; duplicate join keys multiply rows; rare values can be true; learned preprocessing parameters; collection-process evidence and assumptions; repeated transformation yields the same table, but does not prove accuracy.

## Final teaching checklist
Learners should explain the row grain, preserve source data, measure defects, choose justified rules, parse explicitly, retain issue reasons, protect relationships, avoid leakage, test outputs, reconcile changes, and communicate unresolved uncertainty. Domain expertise remains essential.